# S3-01: Tool Use 기초 (도구 정의, 스키마, 첫 도구 호출)
**Skilljar L01-L04: Introducing Tool Use / Project Overview / Tool Functions / Tool Schemas**

## 학습 목표
- Tool Use의 동작 원리를 이해한다
- Python 함수를 도구로 정의하고 JSON Schema로 스키마를 작성한다
- `client.messages.create(tools=...)`로 첫 도구 호출을 수행한다
- 응답에서 `ToolUseBlock`을 추출하고 분석한다

## 사전 준비
1. [Anthropic Console](https://console.anthropic.com)에서 API 키 발급
2. 이 노트북과 같은 폴더에 `.env` 파일 생성:
```
ANTHROPIC_API_KEY="sk-ant-api03-your-key-here"
```

In [28]:
# 패키지 설치
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [29]:
# 환경변수 로드
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# 클라이언트 생성
from pathlib import Path
from dotenv import load_dotenv
from anthropic import Anthropic
import json

# 현재 노트북과 같은 폴더의 .env 로드
load_dotenv(Path().resolve() / ".env", override=True)

client = Anthropic()
model = "claude-sonnet-4-5"
print("설정 완료")

## 1. Tool Use 개념 이해

Tool Use는 Claude가 외부 함수를 **호출 요청**하는 메커니즘이다.

핵심 흐름:
1. 개발자가 도구(함수)를 정의하고 스키마를 Claude에게 전달
2. Claude가 사용자 질문을 분석하여 적절한 도구 호출을 **요청**
3. 개발자 코드가 실제 함수를 실행
4. 실행 결과를 Claude에게 전달
5. Claude가 결과를 바탕으로 자연어 응답 생성

> **중요**: Claude는 도구를 **직접 실행하지 않는다**. 호출 요청만 반환한다.

In [31]:
# 간단한 도구 함수 정의
def get_weather(city: str) -> dict:
    """도시의 현재 날씨를 반환한다 (데모용 가짜 데이터)."""
    # 실제 앱에서는 날씨 API를 호출
    weather_data = {
        "서울": {"temp": 15, "condition": "맑음", "humidity": 45},
        "부산": {"temp": 18, "condition": "구름", "humidity": 60},
        "제주": {"temp": 20, "condition": "비", "humidity": 80},
    }
    return weather_data.get(city, {"temp": 0, "condition": "알 수 없음", "humidity": 0})

# 테스트
print(get_weather("서울"))

{'temp': 15, 'condition': '맑음', 'humidity': 45}


## 2. 도구 스키마 (Tool Schema) 작성

Claude에게 도구를 알려주려면 JSON Schema 형식의 스키마가 필요하다:
- `name`: 도구 이름 (영문, 스네이크 케이스)
- `description`: 도구의 목적과 사용법 (Claude가 읽고 도구 선택)
- `input_schema`: 입력 파라미터 정의 (JSON Schema)

In [32]:
# 날씨 도구의 스키마 정의
weather_tool = {
    "name": "get_weather",
    "description": "지정한 도시의 현재 날씨(온도, 상태, 습도)를 반환한다.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "날씨를 조회할 도시 이름 (예: 서울, 부산, 제주)"
            }
        },
        "required": ["city"]
    }
}

print(json.dumps(weather_tool, indent=2, ensure_ascii=False))

{
  "name": "get_weather",
  "description": "지정한 도시의 현재 날씨(온도, 상태, 습도)를 반환한다.",
  "input_schema": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "날씨를 조회할 도시 이름 (예: 서울, 부산, 제주)"
      }
    },
    "required": [
      "city"
    ]
  }
}


In [33]:
# 도구와 함께 API 호출
response = client.messages.create(
    model=model,
    max_tokens=1024,
    tools=[weather_tool],
    messages=[
        {"role": "user", "content": "서울 날씨가 어때?"}
    ]
)

print(f"stop_reason: {response.stop_reason}")
print(f"content blocks: {len(response.content)}")

for block in response.content:
    print(f"\n--- {block.type} ---")
    if block.type == "text":
        print(f"텍스트: {block.text}")
    elif block.type == "tool_use":
        print(f"도구: {block.name}")
        print(f"입력: {block.input}")
        print(f"ID: {block.id}")

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011Ca2rjULwpXBnpZsBSntwD'}

## 3. ToolUseBlock 분석

Claude가 `stop_reason="tool_use"`로 응답하면, `content`에 `ToolUseBlock`이 포함된다:
- `type`: `"tool_use"`
- `id`: 고유 식별자 (tool_result 매칭용)
- `name`: 호출할 도구 이름
- `input`: 도구에 전달할 인자 (dict)

In [ ]:
# ToolUseBlock 추출 패턴
tool_use_blocks = [b for b in response.content if b.type == "tool_use"]

for block in tool_use_blocks:
    print(f"도구 이름: {block.name}")
    print(f"도구 ID: {block.id}")
    print(f"도구 입력: {json.dumps(block.input, ensure_ascii=False)}")
    
    # 실제 함수 실행
    result = get_weather(**block.input)
    print(f"실행 결과: {result}")

---
## 연습 1: 계산기 도구 정의

간단한 사칙연산 계산기 도구를 정의하세요.

**요구사항:**
1. 함수명: `calculator`
2. 입력: `a` (number), `b` (number), `operation` (string: add/subtract/multiply/divide)
3. 반환: 계산 결과 dict
4. 도구 스키마를 JSON Schema로 작성
5. Claude에게 "23 곱하기 17은?"을 질문하여 도구 호출 확인

In [ ]:
# TODO: 계산기 도구 함수를 작성하세요

def calculator(a: float, b: float, operation: str) -> dict:
    """사칙연산 계산기"""
    pass  # 여기에 구현

# TODO: 도구 스키마를 작성하세요

calculator_tool = {
    "name": "calculator",
    "description": "TODO",
    "input_schema": {
        # TODO
    }
}

# TODO: Claude에게 질문하여 도구 호출 확인

# response = client.messages.create(...)

In [ ]:
# ===== 정답 =====

def calculator(a: float, b: float, operation: str) -> dict:
    """사칙연산 계산기"""
    ops = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else None
    }
    result = ops.get(operation)
    if result is None:
        return {"error": f"잘못된 연산: {operation}"}
    return {"a": a, "b": b, "operation": operation, "result": result}

calculator_tool = {
    "name": "calculator",
    "description": "두 수의 사칙연산(덧셈, 뺄셈, 곱셈, 나눗셈)을 수행한다.",
    "input_schema": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "첫 번째 수"},
            "b": {"type": "number", "description": "두 번째 수"},
            "operation": {
                "type": "string",
                "description": "연산 종류",
                "enum": ["add", "subtract", "multiply", "divide"]
            }
        },
        "required": ["a", "b", "operation"]
    }
}

response = client.messages.create(
    model=model,
    max_tokens=1024,
    tools=[calculator_tool],
    messages=[{"role": "user", "content": "23 곱하기 17은?"}]
)

print(f"stop_reason: {response.stop_reason}")
for block in response.content:
    if block.type == "tool_use":
        print(f"도구: {block.name}, 입력: {block.input}")
        result = calculator(**block.input)
        print(f"결과: {result}")

def verify():
    # 함수 검증
    r1 = calculator(23, 17, "multiply")
    assert r1["result"] == 391, f"23*17은 391이어야 함, got {r1['result']}"
    r2 = calculator(10, 3, "add")
    assert r2["result"] == 13, f"10+3은 13이어야 함, got {r2['result']}"
    # 스키마 검증
    assert calculator_tool["name"] == "calculator"
    assert "input_schema" in calculator_tool
    assert "required" in calculator_tool["input_schema"]
    # API 응답 검증
    assert response.stop_reason == "tool_use"
    print("모든 검증 통과!")

verify()

---
## 연습 2: 도구 스키마의 description 효과

동일한 도구에 대해 **description을 변경**하면 Claude의 도구 사용 결정이 달라지는지 확인하세요.

**요구사항:**
1. 같은 함수에 대해 2가지 다른 description으로 스키마 작성
2. 같은 질문을 던져서 Claude의 도구 호출 여부 비교
3. description이 모호하면 Claude가 도구를 호출하지 않을 수 있음을 확인

In [ ]:
# TODO: description이 다른 2가지 스키마를 작성하세요

# tool_good_desc = { ... }  # 상세한 description
# tool_bad_desc = { ... }   # 모호한 description

In [ ]:
# ===== 정답 =====

def unit_converter(value: float, from_unit: str, to_unit: str) -> dict:
    """단위 변환기"""
    conversions = {
        ("mm", "m"): value / 1000,
        ("m", "mm"): value * 1000,
        ("kN", "N"): value * 1000,
        ("N", "kN"): value / 1000,
        ("MPa", "kPa"): value * 1000,
        ("kPa", "MPa"): value / 1000,
    }
    result = conversions.get((from_unit, to_unit))
    if result is None:
        return {"error": f"변환 불가: {from_unit} -> {to_unit}"}
    return {"value": value, "from": from_unit, "to": to_unit, "result": result}

# 상세한 description
tool_good_desc = {
    "name": "unit_converter",
    "description": (
        "건축공학 단위를 변환한다. 지원: mm<->m, kN<->N, MPa<->kPa. "
        "길이, 하중, 응력의 단위 변환이 필요할 때 사용한다."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "value": {"type": "number", "description": "변환할 값"},
            "from_unit": {"type": "string", "description": "원래 단위"},
            "to_unit": {"type": "string", "description": "변환 대상 단위"}
        },
        "required": ["value", "from_unit", "to_unit"]
    }
}

# 모호한 description
tool_bad_desc = {
    "name": "unit_converter",
    "description": "변환",  # 너무 짧고 모호함
    "input_schema": tool_good_desc["input_schema"]
}

question = "500mm를 m로 변환해줘."

# 좋은 description으로 호출
resp_good = client.messages.create(
    model=model, max_tokens=512,
    tools=[tool_good_desc],
    messages=[{"role": "user", "content": question}]
)
print(f"좋은 desc → stop_reason: {resp_good.stop_reason}")

# 나쁜 description으로 호출
resp_bad = client.messages.create(
    model=model, max_tokens=512,
    tools=[tool_bad_desc],
    messages=[{"role": "user", "content": question}]
)
print(f"나쁜 desc → stop_reason: {resp_bad.stop_reason}")

def verify():
    assert resp_good.stop_reason == "tool_use", "좋은 description은 도구 호출을 유도해야 함"
    print("검증 통과: description이 도구 호출에 영향을 미침을 확인!")

verify()

---
## 연습 3: 다양한 input_schema 타입

JSON Schema의 다양한 타입 (`string`, `number`, `boolean`, `array`)을 사용하는 도구를 정의하세요.

**요구사항:**
1. 함수명: `create_beam_spec`
2. 입력:
   - `beam_id` (string): 보 ID
   - `width` (number): 너비 mm
   - `depth` (number): 깊이 mm
   - `is_seismic` (boolean): 내진설계 적용 여부
   - `rebar_sizes` (array of string): 철근 규격 리스트
3. 스키마를 작성하고 Claude에게 보 사양을 요청하여 도구 호출 확인

In [ ]:
# TODO: create_beam_spec 함수와 스키마를 작성하세요

In [ ]:
# ===== 정답 =====

def create_beam_spec(beam_id: str, width: float, depth: float,
                     is_seismic: bool, rebar_sizes: list) -> dict:
    """보 사양서를 생성한다."""
    return {
        "beam_id": beam_id,
        "section": f"{width}x{depth}",
        "width_mm": width,
        "depth_mm": depth,
        "seismic_design": is_seismic,
        "rebar": rebar_sizes,
        "rebar_count": len(rebar_sizes)
    }

beam_spec_tool = {
    "name": "create_beam_spec",
    "description": "RC 보의 사양서를 생성한다. 보 ID, 단면 치수, 내진설계 여부, 철근 규격을 입력받는다.",
    "input_schema": {
        "type": "object",
        "properties": {
            "beam_id": {"type": "string", "description": "보 부재 ID (예: B1, G2)"},
            "width": {"type": "number", "description": "보 너비 (mm)"},
            "depth": {"type": "number", "description": "보 깊이 (mm)"},
            "is_seismic": {"type": "boolean", "description": "내진설계 적용 여부"},
            "rebar_sizes": {
                "type": "array",
                "description": "사용 철근 규격 리스트",
                "items": {"type": "string"}
            }
        },
        "required": ["beam_id", "width", "depth", "is_seismic", "rebar_sizes"]
    }
}

response = client.messages.create(
    model=model, max_tokens=1024,
    tools=[beam_spec_tool],
    messages=[{"role": "user", "content": "B1 보 사양을 만들어줘. 350x700, 내진설계 적용, 철근은 D25, D22, D19를 사용해."}]
)

print(f"stop_reason: {response.stop_reason}")
for block in response.content:
    if block.type == "tool_use":
        print(f"입력: {json.dumps(block.input, indent=2, ensure_ascii=False)}")
        result = create_beam_spec(**block.input)
        print(f"결과: {json.dumps(result, indent=2, ensure_ascii=False)}")

def verify():
    assert response.stop_reason == "tool_use"
    tool_block = [b for b in response.content if b.type == "tool_use"][0]
    assert tool_block.name == "create_beam_spec"
    assert isinstance(tool_block.input.get("is_seismic"), bool)
    assert isinstance(tool_block.input.get("rebar_sizes"), list)
    print("모든 검증 통과! (string, number, boolean, array 타입 모두 확인)")

verify()

---
## 건축공학 실습: RC 보 모멘트 계산 도구

### 과제: RC 보의 공칭 휨 모멘트 강도를 계산하는 도구를 정의하세요.

**요구사항:**
1. 함수명: `calculate_nominal_moment`
2. 입력: `b` (mm), `d` (mm), `fck` (MPa), `fy` (MPa), `As` (mm2)
3. KDS 14 20 20 등가직사각형 응력 블록 방법 사용
4. beta1은 fck에 따라 결정: fck<=28이면 0.85, 28<fck<=56이면 선형 감소, fck>56이면 0.65
5. 도구 스키마 작성
6. Claude에게 "350x600 보, fck=27MPa, fy=400MPa, 5-D25(2540mm2)의 모멘트를 계산해줘"라고 질문
7. 도구 호출 결과 확인

In [ ]:
# TODO: 모멘트 계산 도구 함수와 스키마를 작성하세요

In [ ]:
# ===== 정답 =====

def calculate_nominal_moment(b: float, d: float, fck: float, fy: float, As: float) -> dict:
    """RC 보의 공칭 휨 모멘트 강도 Mn 계산 (KDS 14 20 20)"""
    # beta1 결정
    if fck <= 28:
        beta1 = 0.85
    elif fck <= 56:
        beta1 = 0.85 - 0.007 * (fck - 28)
    else:
        beta1 = 0.65

    # 등가직사각형 응력 블록
    a = (As * fy) / (0.85 * fck * b)
    c = a / beta1

    # 공칭 모멘트 강도
    Mn = As * fy * (d - a / 2) / 1e6  # kN-m

    # 인장 철근 변형률
    epsilon_t = 0.003 * (d - c) / c

    # 강도감소계수
    if epsilon_t >= 0.005:
        phi = 0.85
    elif epsilon_t <= 0.002:
        phi = 0.65
    else:
        phi = 0.65 + (epsilon_t - 0.002) * (200 / 3)

    return {
        "a_mm": round(a, 1),
        "c_mm": round(c, 1),
        "beta1": round(beta1, 3),
        "Mn_kNm": round(Mn, 1),
        "phi": round(phi, 3),
        "phi_Mn_kNm": round(phi * Mn, 1),
        "epsilon_t": round(epsilon_t, 5),
        "rho": round(As / (b * d), 4)
    }

moment_tool = {
    "name": "calculate_nominal_moment",
    "description": (
        "RC 보의 공칭 휨 모멘트 강도 Mn을 KDS 14 20 20 기준으로 계산한다. "
        "등가직사각형 응력 블록 방법을 사용하며, 강도감소계수 phi도 산정한다."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "b": {"type": "number", "description": "보 너비 (mm)"},
            "d": {"type": "number", "description": "유효 깊이 (mm)"},
            "fck": {"type": "number", "description": "콘크리트 설계기준강도 (MPa)"},
            "fy": {"type": "number", "description": "철근 항복강도 (MPa)"},
            "As": {"type": "number", "description": "인장 철근 단면적 (mm2)"}
        },
        "required": ["b", "d", "fck", "fy", "As"]
    }
}

response = client.messages.create(
    model=model, max_tokens=1024,
    tools=[moment_tool],
    messages=[{"role": "user", "content": "350x600 보, fck=27MPa, fy=400MPa, 5-D25(2540mm2)의 모멘트를 계산해줘."}]
)

print(f"stop_reason: {response.stop_reason}")
for block in response.content:
    if block.type == "tool_use":
        print(f"\n도구 입력: {json.dumps(block.input, indent=2)}")
        result = calculate_nominal_moment(**block.input)
        print(f"\n계산 결과: {json.dumps(result, indent=2, ensure_ascii=False)}")

def verify():
    # 함수 직접 검증
    r = calculate_nominal_moment(b=350, d=600, fck=27, fy=400, As=2540)
    assert 300 < r["Mn_kNm"] < 700, f"Mn이 합리적 범위 밖: {r['Mn_kNm']}"
    assert r["beta1"] == 0.85, f"fck=27일 때 beta1은 0.85여야 함: {r['beta1']}"
    assert 0.005 < r["epsilon_t"], "인장지배 확인"
    assert r["phi"] == 0.85, f"인장지배이면 phi=0.85: {r['phi']}"
    # API 응답 검증
    assert response.stop_reason == "tool_use"
    print(f"모든 검증 통과! Mn = {r['Mn_kNm']} kN-m, phi*Mn = {r['phi_Mn_kNm']} kN-m")

verify()